# 06 · Nebraska replication of the drought-penalty method (pre-registered)

**Question.** Does the notebook 05 drought-penalty method, designed on Iowa, work **unchanged** on Nebraska's mostly-dryland counties?

**Status.** This cell is the full plan, written and committed **before any Nebraska result exists**. Every rule below is fixed in advance. Nothing is tuned after seeing results. Whatever the outcome, it is reported.

## Method (imported, not copied)
- All logic is imported from `soil_planner.soil_tool`: `compute_penalties`, `penalty_ranking_results`, `ranking_summary_table`, and the parameters `WINDOW`, `MIN_NORMAL_YEARS`, `SEV_D2`, `MIN_D2_YEARS_FOR_SCORE`, `TRAIN_YEARS`, `TEST_YEARS`, `N_BOOT`, `BOOT_SEED`, `CI`, `TOP_SHARE`, and the placebo definitions `REAL_TEST`, `PLACEBO_ZERO`, `PLACEBO_ALL`. `soil_tool.py` is not edited.
- `soil_tool.load_iowa()` is hard-wired to Iowa, so this notebook builds the Nebraska input itself from `src.data_loader.load_master()`: rows with `state_alpha == "NE"`, `crop` in `{CORN, SOYBEANS}`, a reported yield (`yield_per_acre` not null), `fips` kept as 5-character text, and passes them through the **same** functions. Corn and soybeans are run separately, as in Iowa.
- **Fixed parameters, identical to Iowa** (read from the module and asserted at run time): `WINDOW = 3` years around the target year for the county's normal; at least `MIN_NORMAL_YEARS = 2` normal years; drought year = `max_drought_severity >= SEV_D2 = 3` (D2+); at least `MIN_D2_YEARS_FOR_SCORE = 3` drought years with a penalty for a score; TRAIN `2000–2015`, TEST `2016–2025`; `N_BOOT = 5000` bootstrap draws with `BOOT_SEED = 20261017`; 95% range from the 2.5th and 97.5th percentiles; top third = `TOP_SHARE = 1/3`; the same three scores (penalty score, drought frequency, soil water storage) and the same two placebos (zero-drought test years; non-D2+ counties in all test years).
- **One deliberate difference, fixed now:** the minimum number of counties for a usable year is **10** instead of Iowa's 15, only because the dryland pool is smaller. It is applied by setting the module attribute `soil_tool.MIN_COUNTIES_PER_YEAR = 10` at run time for the Nebraska run (the file is unchanged; the Iowa comparison is computed first, with its own 15, and the attribute is restored afterwards). That attribute governs both which years count as usable and the per-year evaluation inside `penalty_ranking_results`.
- **"Remove what hit everyone" over dryland counties only.** `compute_penalties` subtracts the statewide median log yield of the rows it receives. Because only Nebraska dryland rows are passed, that median is the median over the dryland counties, as required.

## Dryland definition (fixed)
- From `src.data_loader.load_irrigation()`: Nebraska rows for the crop; for each county, the **mean of `irrigated_share` over the years with a value**. A county is **dryland** if that mean is **below 0.20**.
- Counties with **no irrigation rows for that crop are excluded**, not assumed dryland.
- The irrigation extension mostly ends in 2018. Dryland status is therefore measured on the years available and **assumed to hold afterwards**; the notebook reports the last Nebraska year with data.

## Step 1: feasibility gate (reported before anything else is computed)
Per crop: (a) number of dryland counties; (b) number of dryland counties with a reported yield in **most years, defined as at least 14 of the 26 years**; (c) number of TRAIN years and of TEST years with **at least 10 dryland counties at D2+ with a reported yield**.

**Proceed to Step 2 for a crop only if (a) ≥ 20 and the TEST count in (c) ≥ 3.** Otherwise stop for that crop and report why. Note that (c) counts D2+ counties with a reported yield; the test itself additionally needs a valid penalty for each county-year, so a year that passes the gate can still be dropped inside the test. That is accepted in advance.

## Step 2: replication (crops that pass)
- `compute_penalties(nebraska_dryland_rows, split=True)` (normals never cross 2015/2016), then `penalty_ranking_results(...)` with the threshold set to 10 as above: TRAIN-only scores tested on TEST drought years; outcomes are the **Spearman rank correlation** (signed so positive = correct ordering, chance 0) and the **top-third hit rate** (chance 0.33), each averaged over usable test years with a year-bootstrap 95% range; plus the two placebos.
- Results are shown **side by side with Iowa's** (same functions, Iowa's threshold of 15), with the number of years and of counties each rests on.

## Pre-registered success rule
The method **succeeds** for a crop if, for the **penalty score** on the real test, **the 95% range of the top-third hit rate lies entirely above 0.33 AND the broad placebo (non-D2+ counties, all test years) stays at chance**, meaning its 95% ranges for the penalty score include the chance value for both metrics (0 for Spearman, 0.33 for the top-third hit rate). Any other pattern is reported as **not meeting the rule**. The Spearman result, drought frequency and storage are reported but do not enter the rule.

## Step 3: plain-language summary (written after the results, judged by the rule above)
Did the method meet the rule? Did the placebo stay clean? How much weight can the result bear given the number of years? State that irrigation data mostly ends in 2018 so later dryland status is assumed, not measured. If a range clears chance on few years, note that the size of the effect is likely overstated (Gelman, Hill & Vehtari, *Regression and Other Stories*, §4.4, §16.1). Do not state conclusions the data does not support.

## Scope
Nebraska is **not** added to the app, the priority map or `soil_tool.py`. Nothing under `data/` or `src/` is modified. The notebook runs from the repo root with `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=600 notebooks/06_nebraska_replication.ipynb`.
